# Round 5 Cherry Picking Winners — 微观结构初探

本轮共有 **50 个全新产品**（不能再交易前几轮的资产），分成 **10 组**，每组 5 个：

| ID | Group | Members |
|----|-------|---------|
| 1 | Galaxy Sounds Recorders | DARK_MATTER · BLACK_HOLES · PLANETARY_RINGS · SOLAR_WINDS · SOLAR_FLAMES |
| 2 | Vertical Sleeping Pods | SUEDE · LAMB_WOOL · POLYESTER · NYLON · COTTON |
| 3 | Organic Microchips | CIRCLE · OVAL · SQUARE · RECTANGLE · TRIANGLE |
| 4 | Purification Pebbles | XS · S · M · L · XL |
| 5 | Domestic Robots | VACUUMING · MOPPING · DISHES · LAUNDRY · IRONING |
| 6 | UV-Visors | YELLOW · AMBER · ORANGE · RED · MAGENTA |
| 7 | Instant Translators | SPACE_GRAY · ASTRO_BLACK · ECLIPSE_CHARCOAL · GRAPHITE_MIST · VOID_BLUE |
| 8 | Construction Panels | 1X2 · 2X2 · 1X4 · 2X4 · 4X4 |
| 9 | Liquid Breath Oxygen Shakes | MORNING_BREATH · EVENING_BREATH · MINT · CHOCOLATE · GARLIC |
| 10 | Protein Snack Packs | CHOCOLATE · VANILLA · PISTACHIO · STRAWBERRY · RASPBERRY |

所有产品的 **position limit = 10**。这一轮没有期权 / underlying / voucher 的概念，所以本 notebook 只关心**单产品的盘口微观结构**（depth + trades + spread + activity），不复用 round 3/4 的期权图。

## 中间价定义

**wall_mid**：取 bid 侧挂单量最大的那一档价格 + ask 侧挂单量最大的那一档价格，再除 2。这一列由 `vev_plot.enrich.attach_spread` 自动写到 `ob_wide.wall_mid`。

**fair_price (round 5)**：
- 先算 `wall_mid`
- 统计 6 档报价中 |price - wall_mid| <= 1 的挂单数量
- 恰好 1 档 → fair_price = 该挂单价格
- 0 档或 >=2 档 → fair_price = wall_mid - 0.5
- 若单边/双边缺失 → 按 product 前向填充

normalize 视图一律用 `price - fair_price`。

## 用法

1. 跑下面的 *Setup* 把 round 5 三天数据 load 进 `ctx`。
2. 跑 *Group table* 看一遍编号。
3. 调用 `show_group(<id>)`（id ∈ 1..10），就会画出该组 5 个产品各自的微观结构。50 个产品一次性画太大，所以**每次只看一组**。

## Setup —— 加载 round 5 数据

三天的数据 (day 2/3/4) 拼成连续 x 轴 `global_ts = day*1_000_000 + timestamp`，竖虚线即 day 分界。

In [ ]:
import sys
from pathlib import Path

# 让 round5research/ 下的 notebook 也能 import 项目根目录里的 vev_plot
ROOT = Path.cwd()
if ROOT.name == 'round5research':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from vev_plot import Context, enable_plotly_resample
from vev_plot.plots.depth import plot_product_detail

DATA_DIR = ROOT / 'data' / 'bt'
ROUND = 5
DAYS = None  # None = 自动用 data/bt 下能找到的全部 round 5 day

price_files = sorted(DATA_DIR.glob(f'prices_round_{ROUND}_day_*.csv'))
if not price_files:
    raise FileNotFoundError(f'No round {ROUND} prices files under {DATA_DIR}')
available_days = sorted({int(p.stem.split('_')[-1]) for p in price_files})
days = available_days if DAYS is None else sorted(DAYS)

enable_plotly_resample(max_points=5000)
ctx = Context.from_data_dir(DATA_DIR, days=days, round_num=ROUND)
print(f'round={ROUND} days={ctx.days}  products={len(ctx.products)}  '
      f'ob_rows={ctx.ob_wide.height:,}  trades={ctx.trades.height:,}')

## Group table

10 组的编号 + 成员一览。`MISSING` 列写出当前 csv 里缺失的产品名（应该都为空）。

In [ ]:
import pandas as pd

GROUPS = {
    1:  ('Galaxy Sounds Recorders', [
            'GALAXY_SOUNDS_DARK_MATTER', 'GALAXY_SOUNDS_BLACK_HOLES',
            'GALAXY_SOUNDS_PLANETARY_RINGS', 'GALAXY_SOUNDS_SOLAR_WINDS',
            'GALAXY_SOUNDS_SOLAR_FLAMES']),
    2:  ('Vertical Sleeping Pods', [
            'SLEEP_POD_SUEDE', 'SLEEP_POD_LAMB_WOOL', 'SLEEP_POD_POLYESTER',
            'SLEEP_POD_NYLON', 'SLEEP_POD_COTTON']),
    3:  ('Organic Microchips', [
            'MICROCHIP_CIRCLE', 'MICROCHIP_OVAL', 'MICROCHIP_SQUARE',
            'MICROCHIP_RECTANGLE', 'MICROCHIP_TRIANGLE']),
    4:  ('Purification Pebbles', [
            'PEBBLES_XS', 'PEBBLES_S', 'PEBBLES_M', 'PEBBLES_L', 'PEBBLES_XL']),
    5:  ('Domestic Robots', [
            'ROBOT_VACUUMING', 'ROBOT_MOPPING', 'ROBOT_DISHES',
            'ROBOT_LAUNDRY', 'ROBOT_IRONING']),
    6:  ('UV-Visors', [
            'UV_VISOR_YELLOW', 'UV_VISOR_AMBER', 'UV_VISOR_ORANGE',
            'UV_VISOR_RED', 'UV_VISOR_MAGENTA']),
    7:  ('Instant Translators', [
            'TRANSLATOR_SPACE_GRAY', 'TRANSLATOR_ASTRO_BLACK',
            'TRANSLATOR_ECLIPSE_CHARCOAL', 'TRANSLATOR_GRAPHITE_MIST',
            'TRANSLATOR_VOID_BLUE']),
    8:  ('Construction Panels', [
            'PANEL_1X2', 'PANEL_2X2', 'PANEL_1X4', 'PANEL_2X4', 'PANEL_4X4']),
    9:  ('Liquid Breath Oxygen Shakes', [
            'OXYGEN_SHAKE_MORNING_BREATH', 'OXYGEN_SHAKE_EVENING_BREATH',
            'OXYGEN_SHAKE_MINT', 'OXYGEN_SHAKE_CHOCOLATE',
            'OXYGEN_SHAKE_GARLIC']),
    10: ('Protein Snack Packs', [
            'SNACKPACK_CHOCOLATE', 'SNACKPACK_VANILLA', 'SNACKPACK_PISTACHIO',
            'SNACKPACK_STRAWBERRY', 'SNACKPACK_RASPBERRY']),
}

present = set(ctx.products)
rows = []
for gid, (name, members) in GROUPS.items():
    missing = [m for m in members if m not in present]
    rows.append({
        'id': gid,
        'group': name,
        'members': ', '.join(m.split('_', 1)[1] if '_' in m else m for m in members),
        'missing': ', '.join(missing) if missing else '',
    })
pd.DataFrame(rows).set_index('id')

## `show_group(gid)` —— 看一组 5 个产品

对该组每个产品依次画：

1. **盘口 + 成交散点图（normalize=True）**：y = price − fair_price，y=0 即 fair_price。绿点 bid，红点 ask；右轴浅色虚线是 fair_price。
2. **盘口 + 成交散点图（normalize=False）**：y = 原始价格，浅色实线是 fair_price。

**Trade marker 配色** —— 当 trades csv 给出 `buyer` / `seller` 时（如 round 4 的 Mark XX bot），颜色编码 bot 身份，▲ = 该 bot 在此笔做 buyer，▼ = 该 bot 做 seller。**round 5 的 csv 里 buyer/seller 全部为空**，所以自动 fallback 成 taker 方向推断（仍用 wall_mid 作为比较基准）：

- 紫 ▲ = 主动买（成交价 > wall_mid）
- 黄 ▼ = 主动卖（成交价 < wall_mid）
- 灰 ● = 成交价 = wall_mid（中性方向，仍标出位置和大小）

marker 大小 ∝ 成交 quantity；点开图例可单独 toggle 任一类。

之后画两张组级图：

3. **5 条 wall_mid 叠图**（同 y 轴，看共动 / 离散程度）。
4. **spread_1 时序**（每个产品一条线，越平越好做 MM）。
5. **5 个产品的成交活跃度 heatmap**（rows=product, cols=time bin）。

如果只想看其中一两张，直接调用 `plot_product_detail(ctx, '<symbol>')` 即可。

In [ ]:
from typing import Iterable

import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots


def _group_members(gid: int) -> tuple[str, list[str]]:
    if gid not in GROUPS:
        raise ValueError(f'unknown group id {gid}; valid: {sorted(GROUPS)}')
    name, members = GROUPS[gid]
    present = [m for m in members if m in ctx.products]
    return name, present


def _day_vlines(fig: go.Figure, days: list[int], **kw):
    for d in days[1:]:
        fig.add_vline(x=d * 1_000_000,
                      line=dict(color='rgba(0,0,0,0.35)', width=1, dash='dash'),
                      **kw)


def plot_group_wallmid(gid: int) -> go.Figure:
    name, members = _group_members(gid)
    fig = go.Figure()
    palette = px.colors.qualitative.Set1
    for i, sym in enumerate(members):
        df = (
            ctx.ob_wide.filter(pl.col('product') == sym)
            .select(['global_ts', 'wall_mid']).drop_nulls().sort('global_ts')
        )
        if df.is_empty():
            continue
        fig.add_trace(go.Scattergl(
            x=df['global_ts'].to_list(), y=df['wall_mid'].to_list(),
            mode='lines', name=sym,
            line=dict(width=1, color=palette[i % len(palette)]),
        ))
    _day_vlines(fig, ctx.days)
    fig.update_layout(
        height=420, hovermode='x unified',
        title=f'Group {gid} — {name}: wall_mid 时序',
        xaxis=dict(title='global_ts', showspikes=True, spikemode='across'),
        yaxis=dict(title='wall_mid'),
        legend=dict(orientation='h', y=-0.2),
    )
    return fig


def plot_group_spread(gid: int) -> go.Figure:
    name, members = _group_members(gid)
    fig = go.Figure()
    palette = px.colors.qualitative.Set1
    for i, sym in enumerate(members):
        df = ctx.product_slice(sym).select(['global_ts', 'spread_1']).drop_nulls()
        if df.is_empty():
            continue
        fig.add_trace(go.Scattergl(
            x=df['global_ts'].to_list(), y=df['spread_1'].to_list(),
            mode='lines', name=sym,
            line=dict(width=1, color=palette[i % len(palette)]),
        ))
    _day_vlines(fig, ctx.days)
    fig.update_layout(
        height=360, hovermode='x unified',
        title=f'Group {gid} — {name}: spread_1 (ask1 - bid1) 时序',
        xaxis=dict(title='global_ts'), yaxis=dict(title='spread_1'),
        legend=dict(orientation='h', y=-0.25),
    )
    return fig


def plot_group_activity(gid: int, *, time_bin: int = 20_000,
                        metric: str = 'count') -> go.Figure:
    name, members = _group_members(gid)
    if ctx.trades.is_empty():
        return go.Figure()
    tr = (
        ctx.trades.filter(pl.col('product').is_in(members))
        .with_columns(((pl.col('global_ts') // time_bin) * time_bin).alias('ts_bin'))
    )
    if metric == 'count':
        agg = tr.group_by(['product', 'ts_bin']).len().rename({'len': 'val'})
    else:
        agg = tr.group_by(['product', 'ts_bin']).agg(pl.col('quantity').sum().alias('val'))
    if agg.is_empty():
        return go.Figure().update_layout(title=f'Group {gid} — {name}: no trades')
    pivot = agg.pivot(index='product', on='ts_bin', values='val')
    ts_cols = sorted([c for c in pivot.columns if c != 'product'], key=lambda x: int(x))
    z_dict = {row['product']: [row[c] for c in ts_cols] for row in pivot.iter_rows(named=True)}
    matrix = np.array([z_dict.get(p, [None] * len(ts_cols)) for p in members], dtype=float)
    fig = go.Figure(go.Heatmap(
        z=matrix, x=[int(c) for c in ts_cols], y=members,
        colorscale='Viridis',
        hovertemplate=f't_bin=%{{x}}<br>product=%{{y}}<br>{metric}=%{{z}}<extra></extra>',
    ))
    for d in ctx.days[1:]:
        fig.add_vline(x=d * 1_000_000, line=dict(color='white', width=1, dash='dash'))
    fig.update_layout(
        height=320,
        title=f'Group {gid} — {name}: trade activity (bin={time_bin}, metric={metric})',
    )
    return fig


def show_group(gid: int, *,
               normalize: tuple[bool, ...] = (True, False),
               trade_volume_filter: Iterable[float] | None = None,
               detail_height: int = 520) -> None:
    """画第 gid 组（5 个产品）的全部微观结构图。

    normalize: 默认两种 y 轴口径都画；传 (True,) 或 (False,) 只画一种。
    trade_volume_filter: 仅显示 quantity 在该集合中的成交（None = 全部）。
    """
    name, members = _group_members(gid)
    print(f'>>> group {gid}: {name}  ({len(members)} products)')
    plot_group_wallmid(gid).show()
    plot_group_spread(gid).show()
    plot_group_activity(gid).show()
    for sym in members:
        for norm in normalize:
            plot_product_detail(
                ctx, sym,
                normalize=norm,
                trade_volume_filter=trade_volume_filter,
                height=detail_height,
            ).show()

## 实际使用 —— 输入组号即可

默认看第 1 组（Galaxy Sounds Recorders）。改成别的组号重跑该 cell：`show_group(2)`、`show_group(7)` 等。

In [ ]:
# show_group(1)

其它组示例（取消注释跑）：

In [ ]:
# show_group(2)
# show_group(3)
# show_group(4)
# show_group(5)
# show_group(6)
# show_group(7)
# show_group(8)
# show_group(9)
# show_group(10)

# 只看 normalized 视图：
# show_group(3, normalize=(True,))

# 只看某个产品（绕过 group）：
# plot_product_detail(ctx, 'PANEL_2X4', normalize=True).show()

In [ ]:

show_group(6, normalize=(True,))

In [ ]:

# show_group(6, normalize=(False,))